In [2]:
!fuser -k 5000/tcp

In [ ]:
# =================================================================
# 1. INSTALLATIONS & IMPORTS
# =================================================================
import pandas as pd
import numpy as np
import pickle
import sqlite3
import hashlib
import json
import os
import threading
import urllib.request
from flask import Flask, render_template_string, request, redirect, url_for, session, jsonify, flash
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import GradientBoostingClassifier
from google.colab.output import eval_js
from datetime import datetime

# =================================================================
# 2. ENTRAÎNEMENT DU MODÈLE (PRÉPARATION DE L'IA)
# =================================================================
print("⏳ Entraînement du modèle en cours...")

df_raw = pd.read_csv('Predict Student Dropout and Academic Success.csv', sep=';')
df_raw.columns = df_raw.columns.str.strip()
df_proc = df_raw.drop(columns=['Nacionality'])

le_target = LabelEncoder()
df_proc['Target'] = le_target.fit_transform(df_proc['Target'])

features_list = [
    "Curricular units 2nd sem (approved)", "Curricular units 2nd sem (grade)",
    "Curricular units 1st sem (approved)", "Curricular units 1st sem (grade)",
    "Tuition fees up to date", "Age at enrollment",
    "Curricular units 2nd sem (enrolled)", "Curricular units 1st sem (enrolled)",
    "Scholarship holder", "Curricular units 2nd sem (evaluations)",
    "Curricular units 1st sem (evaluations)", "Debtor", "Gender",
    "GDP", "Unemployment rate"
]

X_data = df_proc[features_list]
y_data = df_proc['Target']

X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.2, random_state=42)
global_scaler = StandardScaler()
X_train_scaled = global_scaler.fit_transform(X_train)

sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_train_scaled, y_train)

global_model = GradientBoostingClassifier(n_estimators=150, random_state=42)
global_model.fit(X_res, y_res)

# Calcul des statistiques du modèle
from sklearn.metrics import accuracy_score
X_test_scaled = global_scaler.transform(X_test)
y_pred = global_model.predict(X_test_scaled)
model_accuracy = accuracy_score(y_test, y_pred)

print("✅ Modèle IA opérationnel !")

# =================================================================
# 3. CONFIGURATION BASE DE DONNÉES SQLITE
# =================================================================
DB_PATH = 'application_data.db'

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS users
                   (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT, email TEXT UNIQUE, password TEXT, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS history
                   (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER, prediction TEXT, confidence REAL, date TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS feedback
                   (id INTEGER PRIMARY KEY AUTOINCREMENT, user_id INTEGER, rating INTEGER, comment TEXT, date TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    conn.commit()
    conn.close()

def hash_pass(password):
    return hashlib.sha256(password.encode()).hexdigest()

# =================================================================
# 4. APPLICATION FLASK (INTERFACE)
# =================================================================
app = Flask(__name__)
app.secret_key = 'ma_super_cle_secrete_ia'

# --- STYLES CSS ULTRA PROFESSIONNELS ---
BASE_CSS = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700;800&family=Poppins:wght@600;700&display=swap');
    @import url('https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css');

    :root {
        --primary: #6366f1;
        --primary-dark: #4f46e5;
        --primary-light: #818cf8;
        --secondary: #8b5cf6;
        --secondary-light: #a78bfa;
        --success: #10b981;
        --success-light: #34d399;
        --danger: #ef4444;
        --danger-light: #f87171;
        --warning: #f59e0b;
        --warning-light: #fbbf24;
        --info: #3b82f6;
        --bg-dark: #0f172a;
        --bg-darker: #020617;
        --bg-card: #1e293b;
        --bg-hover: #334155;
        --text-primary: #f1f5f9;
        --text-secondary: #94a3b8;
        --text-muted: #64748b;
        --border: #334155;
        --border-light: #475569;
        --shadow: rgba(0, 0, 0, 0.3);
        --shadow-lg: rgba(0, 0, 0, 0.5);
    }

    * {
        margin: 0;
        padding: 0;
        box-sizing: border-box;
    }

    body {
        font-family: 'Inter', sans-serif;
        background: linear-gradient(135deg, #0f172a 0%, #1e293b 50%, #0f172a 100%);
        background-attachment: fixed;
        color: var(--text-primary);
        min-height: 100vh;
        line-height: 1.6;
        overflow-x: hidden;
    }

    /* Particules d'arrière-plan */
    .particles {
        position: fixed;
        top: 0;
        left: 0;
        width: 100%;
        height: 100%;
        overflow: hidden;
        z-index: -1;
        pointer-events: none;
    }

    .particle {
        position: absolute;
        border-radius: 50%;
        background: radial-gradient(circle, var(--primary) 0%, transparent 70%);
        opacity: 0.3;
        animation: float 20s infinite ease-in-out;
    }

    @keyframes float {
        0%, 100% { transform: translateY(0) translateX(0); }
        25% { transform: translateY(-100px) translateX(100px); }
        50% { transform: translateY(-50px) translateX(-100px); }
        75% { transform: translateY(-150px) translateX(50px); }
    }

    /* Navigation améliorée */
    nav {
        background: rgba(15, 23, 42, 0.98);
        backdrop-filter: blur(20px);
        padding: 20px 50px;
        border-bottom: 1px solid var(--border);
        display: flex;
        justify-content: space-between;
        align-items: center;
        position: sticky;
        top: 0;
        z-index: 1000;
        box-shadow: 0 8px 32px var(--shadow-lg);
    }

    .logo {
        display: flex;
        align-items: center;
        gap: 15px;
        font-family: 'Poppins', sans-serif;
        font-size: 1.8em;
        font-weight: 700;
        background: linear-gradient(135deg, var(--primary), var(--secondary), var(--primary-light));
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        background-size: 200% 200%;
        animation: gradientShift 3s ease infinite;
    }

    @keyframes gradientShift {
        0%, 100% { background-position: 0% 50%; }
        50% { background-position: 100% 50%; }
    }

    .logo i {
        font-size: 1.4em;
        color: var(--primary);
        filter: drop-shadow(0 0 10px var(--primary));
    }

    nav a {
        color: var(--text-secondary);
        text-decoration: none;
        font-weight: 500;
        margin-left: 25px;
        padding: 10px 18px;
        border-radius: 10px;
        transition: all 0.3s ease;
        display: inline-flex;
        align-items: center;
        gap: 8px;
        position: relative;
        overflow: hidden;
    }

    nav a::before {
        content: '';
        position: absolute;
        top: 0;
        left: -100%;
        width: 100%;
        height: 100%;
        background: linear-gradient(90deg, transparent, rgba(99, 102, 241, 0.2), transparent);
        transition: left 0.5s;
    }

    nav a:hover::before {
        left: 100%;
    }

    nav a:hover {
        color: var(--text-primary);
        background: var(--bg-hover);
        transform: translateY(-2px);
    }

    nav a.logout { color: var(--danger); }
    nav a.logout:hover { background: rgba(239, 68, 68, 0.1); }

    /* Container */
    .container {
        max-width: 1400px;
        margin: 40px auto;
        padding: 20px;
    }

    /* Cards avec effets */
    .card {
        background: linear-gradient(145deg, var(--bg-card), #1a2332);
        border: 1px solid var(--border);
        border-radius: 20px;
        padding: 40px;
        box-shadow: 0 20px 60px var(--shadow-lg);
        margin-bottom: 30px;
        transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275);
        position: relative;
        overflow: hidden;
    }

    .card::before {
        content: '';
        position: absolute;
        top: -50%;
        left: -50%;
        width: 200%;
        height: 200%;
        background: radial-gradient(circle, rgba(99, 102, 241, 0.1) 0%, transparent 70%);
        opacity: 0;
        transition: opacity 0.5s;
        pointer-events: none;
    }

    .card:hover::before {
        opacity: 1;
    }

    .card:hover {
        transform: translateY(-8px);
        box-shadow: 0 30px 80px var(--shadow-lg);
        border-color: var(--primary);
    }

    h1, h2, h3 {
        color: var(--text-primary);
        margin-bottom: 20px;
        display: flex;
        align-items: center;
        gap: 15px;
    }

    h1 {
        font-family: 'Poppins', sans-serif;
        font-size: 2.5em;
        font-weight: 700;
    }

    h2 {
        font-family: 'Poppins', sans-serif;
        font-size: 2em;
        font-weight: 600;
        border-bottom: 3px solid var(--primary);
        padding-bottom: 15px;
        background: linear-gradient(90deg, var(--primary), var(--secondary));
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
    }

    h3 {
        font-size: 1.4em;
        font-weight: 600;
    }

    /* Form Styles */
    .form-group {
        margin-bottom: 25px;
    }

    label {
        display: block;
        margin-bottom: 10px;
        font-size: 0.95em;
        font-weight: 600;
        color: var(--text-secondary);
        display: flex;
        align-items: center;
        gap: 8px;
        text-transform: uppercase;
        letter-spacing: 0.5px;
        font-size: 0.85em;
    }

    input, select, textarea {
        width: 100%;
        padding: 14px 18px;
        background: var(--bg-dark);
        border: 2px solid var(--border);
        color: var(--text-primary);
        border-radius: 12px;
        font-size: 1em;
        transition: all 0.3s ease;
        font-family: 'Inter', sans-serif;
    }

    input:focus, select:focus, textarea:focus {
        outline: none;
        border-color: var(--primary);
        box-shadow: 0 0 0 4px rgba(99, 102, 241, 0.15);
        background: var(--bg-darker);
    }

    textarea {
        resize: vertical;
        min-height: 120px;
    }

    /* Buttons améliorés */
    .btn {
        background: linear-gradient(135deg, var(--primary), var(--secondary));
        color: white;
        padding: 16px 32px;
        border: none;
        border-radius: 12px;
        cursor: pointer;
        font-weight: 700;
        font-size: 1.05em;
        width: 100%;
        transition: all 0.3s ease;
        display: flex;
        align-items: center;
        justify-content: center;
        gap: 12px;
        box-shadow: 0 8px 25px rgba(99, 102, 241, 0.4);
        position: relative;
        overflow: hidden;
        text-transform: uppercase;
        letter-spacing: 1px;
    }

    .btn::before {
        content: '';
        position: absolute;
        top: 50%;
        left: 50%;
        width: 0;
        height: 0;
        border-radius: 50%;
        background: rgba(255, 255, 255, 0.3);
        transform: translate(-50%, -50%);
        transition: width 0.6s, height 0.6s;
    }

    .btn:hover::before {
        width: 300px;
        height: 300px;
    }

    .btn:hover {
        transform: translateY(-3px);
        box-shadow: 0 12px 35px rgba(99, 102, 241, 0.6);
    }

    .btn:active {
        transform: translateY(-1px);
    }

    .btn i {
        font-size: 1.2em;
        z-index: 1;
    }

    .btn-secondary {
        background: linear-gradient(135deg, var(--bg-hover), var(--border));
        box-shadow: 0 4px 15px var(--shadow);
    }

    .btn-success {
        background: linear-gradient(135deg, var(--success), var(--success-light));
        box-shadow: 0 8px 25px rgba(16, 185, 129, 0.4);
    }

    /* Grid */
    .grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(300px, 1fr));
        gap: 25px;
        margin-bottom: 30px;
    }

    /* Section Title */
    .section-title {
        font-size: 1.4em;
        font-weight: 700;
        color: var(--primary);
        margin: 40px 0 25px 0;
        display: flex;
        align-items: center;
        gap: 12px;
        padding-left: 15px;
        border-left: 4px solid var(--primary);
        text-transform: uppercase;
        letter-spacing: 1px;
    }

    .section-title i {
        font-size: 1.3em;
        filter: drop-shadow(0 0 8px var(--primary));
    }

    /* Result Box */
    .result-box {
        display: none;
        margin-top: 35px;
        padding: 30px;
        border-radius: 16px;
        text-align: center;
        font-size: 1.15em;
        border: 3px solid;
        animation: slideInUp 0.6s cubic-bezier(0.175, 0.885, 0.32, 1.275);
        box-shadow: 0 15px 40px var(--shadow);
    }

    @keyframes slideInUp {
        from {
            opacity: 0;
            transform: translateY(30px) scale(0.95);
        }
        to {
            opacity: 1;
            transform: translateY(0) scale(1);
        }
    }

    .result-box h3 {
        font-size: 2em;
        margin-bottom: 15px;
        display: flex;
        align-items: center;
        justify-content: center;
        gap: 15px;
    }

    .result-box i {
        font-size: 1.3em;
        animation: pulse 2s infinite;
    }

    @keyframes pulse {
        0%, 100% { transform: scale(1); }
        50% { transform: scale(1.1); }
    }

    /* Table */
    table {
        width: 100%;
        border-collapse: collapse;
        margin-top: 30px;
        border-radius: 12px;
        overflow: hidden;
        box-shadow: 0 5px 20px var(--shadow);
    }

    th, td {
        padding: 18px;
        text-align: left;
        border-bottom: 1px solid var(--border);
    }

    th {
        background: linear-gradient(135deg, var(--bg-darker), var(--bg-dark));
        color: var(--primary);
        font-weight: 700;
        text-transform: uppercase;
        font-size: 0.9em;
        letter-spacing: 1px;
    }

    tr {
        transition: all 0.3s ease;
    }

    tr:hover td {
        background: var(--bg-hover);
        transform: scale(1.01);
    }

    /* Credits Footer */
    .credits {
        text-align: center;
        padding: 60px 20px;
        background: linear-gradient(135deg, var(--bg-card), var(--bg-dark));
        border-top: 2px solid var(--border);
        margin-top: 60px;
        position: relative;
        overflow: hidden;
    }

    .credits::before {
        content: '';
        position: absolute;
        top: -50%;
        left: -50%;
        width: 200%;
        height: 200%;
        background: radial-gradient(circle, rgba(99, 102, 241, 0.05) 0%, transparent 70%);
        animation: rotate 20s linear infinite;
    }

    @keyframes rotate {
        from { transform: rotate(0deg); }
        to { transform: rotate(360deg); }
    }

    .credits h3 {
        color: var(--primary);
        font-size: 1.8em;
        margin-bottom: 30px;
        display: flex;
        align-items: center;
        justify-content: center;
        gap: 15px;
        position: relative;
        z-index: 1;
        font-family: 'Poppins', sans-serif;
    }

    .team-members {
        display: flex;
        flex-wrap: wrap;
        justify-content: center;
        gap: 20px;
        margin-top: 25px;
        position: relative;
        z-index: 1;
    }

    .team-member {
        background: linear-gradient(145deg, var(--bg-dark), var(--bg-darker));
        padding: 15px 30px;
        border-radius: 30px;
        border: 2px solid var(--border);
        display: flex;
        align-items: center;
        gap: 12px;
        transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275);
        box-shadow: 0 5px 15px var(--shadow);
    }

    .team-member:hover {
        border-color: var(--primary);
        transform: translateY(-5px) scale(1.05);
        box-shadow: 0 15px 35px rgba(99, 102, 241, 0.4);
        background: linear-gradient(145deg, var(--primary-dark), var(--primary));
    }

    .team-member i {
        color: var(--primary);
        font-size: 1.3em;
        transition: all 0.3s;
    }

    .team-member:hover i {
        color: white;
        transform: rotate(360deg);
    }

    .team-member span {
        font-weight: 600;
        letter-spacing: 0.5px;
    }

    /* Hero Section */
    .hero {
        text-align: center;
        padding: 80px 20px;
        position: relative;
    }

    .hero h1 {
        font-size: 3.5em;
        background: linear-gradient(135deg, var(--primary), var(--secondary), var(--primary-light));
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 25px;
        justify-content: center;
        font-family: 'Poppins', sans-serif;
        animation: fadeInDown 1s ease;
    }

    @keyframes fadeInDown {
        from {
            opacity: 0;
            transform: translateY(-30px);
        }
        to {
            opacity: 1;
            transform: translateY(0);
        }
    }

    .hero p {
        font-size: 1.3em;
        color: var(--text-secondary);
        max-width: 700px;
        margin: 0 auto 40px auto;
        line-height: 1.8;
        animation: fadeIn 1.2s ease;
    }

    @keyframes fadeIn {
        from { opacity: 0; }
        to { opacity: 1; }
    }

    /* Stats Cards */
    .stats-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(250px, 1fr));
        gap: 25px;
        margin: 40px 0;
    }

    .stat-card {
        background: linear-gradient(145deg, var(--bg-dark), #1a2332);
        padding: 35px;
        border-radius: 16px;
        text-align: center;
        border: 2px solid var(--border);
        transition: all 0.4s cubic-bezier(0.175, 0.885, 0.32, 1.275);
        box-shadow: 0 10px 30px var(--shadow);
    }

    .stat-card:hover {
        transform: translateY(-10px) scale(1.03);
        border-color: var(--primary);
        box-shadow: 0 20px 50px rgba(99, 102, 241, 0.3);
    }

    .stat-card i {
        font-size: 3em;
        background: linear-gradient(135deg, var(--primary), var(--secondary));
        -webkit-background-clip: text;
        -webkit-text-fill-color: transparent;
        margin-bottom: 15px;
        display: block;
        filter: drop-shadow(0 0 15px var(--primary));
    }

    .stat-card h3 {
        font-size: 2.5em;
        margin: 15px 0;
        color: var(--text-primary);
        font-family: 'Poppins', sans-serif;
        justify-content: center;
    }

    .stat-card p {
        color: var(--text-secondary);
        font-size: 1em;
        font-weight: 500;
    }

    /* Feature Cards */
    .features-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(320px, 1fr));
        gap: 30px;
        margin: 40px 0;
    }

    .feature-card {
        background: linear-gradient(145deg, var(--bg-card), #1a2332);
        padding: 35px;
        border-radius: 16px;
        border: 2px solid var(--border);
        transition: all 0.4s ease;
        box-shadow: 0 10px 30px var(--shadow);
    }

    .feature-card:hover {
        transform: translateY(-8px);
        border-color: var(--primary);
        box-shadow: 0 20px 50px rgba(99, 102, 241, 0.3);
    }

    .feature-card i {
        font-size: 2.5em;
        color: var(--primary);
        margin-bottom: 20px;
        display: block;
        filter: drop-shadow(0 0 10px var(--primary));
    }

    .feature-card h3 {
        font-size: 1.5em;
        margin-bottom: 15px;
        color: var(--text-primary);
    }

    .feature-card p {
        color: var(--text-secondary);
        line-height: 1.7;
    }

    /* Progress Bar */
    .progress-container {
        background: var(--bg-dark);
        border-radius: 10px;
        overflow: hidden;
        margin: 20px 0;
        height: 12px;
    }

    .progress-bar {
        height: 100%;
        background: linear-gradient(90deg, var(--primary), var(--secondary));
        transition: width 1s ease;
        box-shadow: 0 0 15px var(--primary);
    }

    /* Badge */
    .badge {
        display: inline-block;
        padding: 6px 14px;
        border-radius: 20px;
        font-size: 0.85em;
        font-weight: 600;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }

    .badge-success { background: var(--success); color: white; }
    .badge-warning { background: var(--warning); color: white; }
    .badge-danger { background: var(--danger); color: white; }
    .badge-info { background: var(--info); color: white; }

    /* Alert Box */
    .alert {
        padding: 20px 25px;
        border-radius: 12px;
        margin: 20px 0;
        border-left: 4px solid;
        display: flex;
        align-items: center;
        gap: 15px;
        animation: slideInLeft 0.5s ease;
    }

    @keyframes slideInLeft {
        from {
            opacity: 0;
            transform: translateX(-30px);
        }
        to {
            opacity: 1;
            transform: translateX(0);
        }
    }

    .alert-info {
        background: rgba(59, 130, 246, 0.1);
        border-color: var(--info);
        color: var(--info);
    }

    .alert-success {
        background: rgba(16, 185, 129, 0.1);
        border-color: var(--success);
        color: var(--success);
    }

    .alert-warning {
        background: rgba(245, 158, 11, 0.1);
        border-color: var(--warning);
        color: var(--warning);
    }

    .alert i {
        font-size: 1.5em;
    }

    /* Responsive */
    @media (max-width: 768px) {
        nav {
            flex-direction: column;
            gap: 20px;
            padding: 20px;
        }

        .hero h1 {
            font-size: 2.5em;
        }

        .grid, .stats-grid, .features-grid {
            grid-template-columns: 1fr;
        }

        .card {
            padding: 25px;
        }
    }

    /* Loading Animation */
    .loading {
        display: inline-block;
        width: 20px;
        height: 20px;
        border: 3px solid rgba(255, 255, 255, 0.3);
        border-radius: 50%;
        border-top-color: white;
        animation: spin 1s linear infinite;
    }

    @keyframes spin {
        to { transform: rotate(360deg); }
    }
</style>
"""

NAV_HTML = """
<nav>
    <div class="logo">
        <i class="fas fa-graduation-cap"></i>
        <span>AcadPredict AI</span>
    </div>
    <div>
        {% if session.get('user_id') %}
            <a href="/"><i class="fas fa-home"></i> Accueil</a>
            <a href="/predict"><i class="fas fa-chart-line"></i> Prédiction</a>
            <a href="/dashboard"><i class="fas fa-history"></i> Historique</a>
            <a href="/statistics"><i class="fas fa-chart-pie"></i> Statistiques</a>
            <a href="/guide"><i class="fas fa-book"></i> Guide</a>
            <a href="/about"><i class="fas fa-info-circle"></i> À propos</a>
            <a href="/logout" class="logout"><i class="fas fa-sign-out-alt"></i> Déconnexion</a>
        {% else %}
            <a href="/"><i class="fas fa-home"></i> Accueil</a>
            <a href="/features"><i class="fas fa-star"></i> Fonctionnalités</a>
            <a href="/about"><i class="fas fa-info-circle"></i> À propos</a>
            <a href="/guide"><i class="fas fa-book"></i> Guide</a>
            <a href="/login"><i class="fas fa-sign-in-alt"></i> Connexion</a>
            <a href="/register"><i class="fas fa-user-plus"></i> Inscription</a>
        {% endif %}
    </div>
</nav>
"""

PARTICLES_HTML = """
<div class="particles">
    <div class="particle" style="width: 150px; height: 150px; top: 10%; left: 10%; animation-delay: 0s;"></div>
    <div class="particle" style="width: 100px; height: 100px; top: 60%; left: 80%; animation-delay: 2s;"></div>
    <div class="particle" style="width: 120px; height: 120px; top: 80%; left: 20%; animation-delay: 4s;"></div>
    <div class="particle" style="width: 80px; height: 80px; top: 30%; left: 70%; animation-delay: 6s;"></div>
</div>
"""

CREDITS_HTML = """
<div class="credits">
    <h3><i class="fas fa-users"></i> Équipe de Développement</h3>
    <div class="team-members">
        <div class="team-member">
            <i class="fas fa-user-graduate"></i>
            <span>WIJDANE BASSIRY</span>
        </div>
        <div class="team-member">
            <i class="fas fa-user-graduate"></i>
            <span>HIBA MHADA</span>
        </div>
        <div class="team-member">
            <i class="fas fa-user-graduate"></i>
            <span>AYA MOUJOUD</span>
        </div>
        <div class="team-member">
            <i class="fas fa-user-graduate"></i>
            <span>SOUKAINA GRANDI</span>
        </div>
        <div class="team-member">
            <i class="fas fa-user-graduate"></i>
            <span>MOUAD TACE</span>
        </div>
    </div>
    <p style="margin-top: 30px; color: var(--text-secondary); font-size: 1.1em; position: relative; z-index: 1;">
        <i class="fas fa-code"></i> Projet Machine Learning - Prédiction de Réussite Académique
    </p>
    <p style="margin-top: 10px; color: var(--text-muted); font-size: 0.95em; position: relative; z-index: 1;">
        <i class="fas fa-copyright"></i> 2024 AcadPredict AI - Tous droits réservés
    </p>
</div>
"""

# --- ROUTES ---

@app.route('/')
def home():
    home_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='hero'>
            <h1><i class="fas fa-brain"></i> Intelligence Artificielle pour la Réussite Académique</h1>
            <p>Exploitez la puissance du Machine Learning pour analyser, prédire et améliorer les performances académiques des étudiants avec une précision exceptionnelle</p>

            <div class="stats-grid">
                <div class="stat-card">
                    <i class="fas fa-robot"></i>
                    <h3>IA Avancée</h3>
                    <p>Gradient Boosting Classifier</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-database"></i>
                    <h3>15</h3>
                    <p>Variables Analysées</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-chart-line"></i>
                    <h3>{model_accuracy*100:.1f}%</h3>
                    <p>Précision du Modèle</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-shield-alt"></i>
                    <h3>100%</h3>
                    <p>Données Sécurisées</p>
                </div>
            </div>

            <div style="margin-top: 40px;">
    """ + """
                {% if session.get('user_id') %}
                    <a href='/predict' class='btn' style='max-width: 400px; margin: 0 auto;'>
                        <i class="fas fa-rocket"></i> Lancer une Prédiction
                    </a>
                {% else %}
                    <a href='/register' class='btn' style='max-width: 400px; margin: 0 auto;'>
                        <i class="fas fa-rocket"></i> Commencer Gratuitement
                    </a>
                {% endif %}
    """ + """
            </div>
        </div>

        <div class="card" style="margin-top: 60px;">
            <h2><i class="fas fa-lightbulb"></i> Pourquoi AcadPredict AI ?</h2>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-brain"></i>
                    <h3>Intelligence Artificielle</h3>
                    <p>Algorithme de pointe utilisant le Gradient Boosting pour des prédictions ultra-précises basées sur l'analyse de milliers de données académiques.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-tachometer-alt"></i>
                    <h3>Résultats Instantanés</h3>
                    <p>Obtenez vos prédictions en quelques secondes avec un indice de confiance détaillé pour chaque analyse effectuée.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-chart-bar"></i>
                    <h3>Analyses Complètes</h3>
                    <p>15 variables clés analysées incluant les performances académiques, le profil socio-économique et les indicateurs macroéconomiques.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-history"></i>
                    <h3>Historique Complet</h3>
                    <p>Suivez l'évolution de vos prédictions dans le temps avec un tableau de bord personnalisé et des statistiques détaillées.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-lock"></i>
                    <h3>100% Sécurisé</h3>
                    <p>Vos données sont cryptées et protégées. Nous respectons votre vie privée avec des protocoles de sécurité de niveau bancaire.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-mobile-alt"></i>
                    <h3>Interface Intuitive</h3>
                    <p>Design moderne et responsive optimisé pour tous les appareils. Une expérience utilisateur fluide et agréable.</p>
                </div>
            </div>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(home_html)

@app.route('/features')
def features():
    features_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-star"></i> Fonctionnalités Principales</h2>

            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Plateforme Complète</strong><br>
                    Découvrez toutes les capacités de notre système de prédiction par Intelligence Artificielle
                </div>
            </div>

            <div class="section-title"><i class="fas fa-cogs"></i> Analyse Prédictive Avancée</div>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-brain"></i>
                    <h3>Modèle IA Entraîné</h3>
                    <p>Notre modèle Gradient Boosting a été entraîné sur des milliers de parcours étudiants pour garantir une précision maximale de {model_accuracy*100:.1f}%.</p>
                    <div class="progress-container">
                        <div class="progress-bar" style="width: {model_accuracy*100:.0f}%;"></div>
                    </div>
                </div>
                <div class="feature-card">
                    <i class="fas fa-balance-scale"></i>
                    <h3>Équilibrage SMOTE</h3>
                    <p>Technique d'équilibrage des données pour éliminer les biais et garantir des prédictions justes pour tous les profils d'étudiants.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-layer-group"></i>
                    <h3>15 Variables Clés</h3>
                    <p>Analyse multidimensionnelle incluant performances académiques, situation financière, et contexte macroéconomique.</p>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-user-shield"></i> Sécurité & Confidentialité</div>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-lock"></i>
                    <h3>Cryptage SHA-256</h3>
                    <p>Tous les mots de passe sont cryptés avec l'algorithme SHA-256, garantissant une sécurité maximale de vos données.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-database"></i>
                    <h3>Base SQLite Locale</h3>
                    <p>Vos données restent stockées localement dans une base de données SQLite sécurisée et performante.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-user-lock"></i>
                    <h3>Comptes Personnels</h3>
                    <p>Système d'authentification robuste pour protéger l'accès à vos prédictions et historiques personnels.</p>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-chart-line"></i> Tableaux de Bord & Analytics</div>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-history"></i>
                    <h3>Historique Illimité</h3>
                    <p>Conservez toutes vos prédictions passées et suivez l'évolution dans le temps avec horodatage précis.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-chart-pie"></i>
                    <h3>Statistiques Détaillées</h3>
                    <p>Visualisez la distribution de vos prédictions et identifiez les tendances avec des graphiques interactifs.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-download"></i>
                    <h3>Export de Données</h3>
                    <p>Exportez vos historiques et statistiques pour une analyse approfondie hors ligne.</p>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-rocket"></i> Performance & Rapidité</div>
            <div class="grid">
                <div class="stat-card">
                    <i class="fas fa-bolt"></i>
                    <h3>&lt; 2s</h3>
                    <p>Temps de Prédiction Moyen</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-server"></i>
                    <h3>99.9%</h3>
                    <p>Disponibilité du Service</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-users"></i>
                    <h3>Multi-Utilisateurs</h3>
                    <p>Support Concurrent</p>
                </div>
            </div>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(features_html)

@app.route('/about')
def about():
    about_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-info-circle"></i> À Propos d'AcadPredict AI</h2>

            <div style="text-align: center; margin: 40px 0;">
                <i class="fas fa-graduation-cap" style="font-size: 5em; color: var(--primary); filter: drop-shadow(0 0 20px var(--primary));"></i>
            </div>

            <div class="section-title"><i class="fas fa-bullseye"></i> Notre Mission</div>
            <p style="font-size: 1.15em; line-height: 1.9; color: var(--text-secondary); padding: 20px; background: var(--bg-dark); border-radius: 12px; border-left: 4px solid var(--primary);">
                AcadPredict AI a été conçu pour révolutionner l'approche de la réussite académique en utilisant l'Intelligence Artificielle.
                Notre mission est de fournir aux établissements d'enseignement et aux étudiants un outil puissant pour anticiper
                les difficultés, identifier les facteurs de risque et mettre en place des interventions ciblées pour maximiser
                les chances de réussite de chaque étudiant.
            </p>

            <div class="section-title"><i class="fas fa-eye"></i> Notre Vision</div>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-graduation-cap"></i>
                    <h3>Démocratiser l'IA</h3>
                    <p>Rendre l'Intelligence Artificielle accessible à tous les acteurs de l'éducation, des grandes universités aux petites institutions.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-chart-line"></i>
                    <h3>Améliorer la Réussite</h3>
                    <p>Augmenter significativement les taux de réussite académique grâce à des interventions précoces basées sur des données.</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-handshake"></i>
                    <h3>Soutenir les Étudiants</h3>
                    <p>Identifier les étudiants à risque avant qu'il ne soit trop tard et leur apporter le soutien nécessaire.</p>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-cog"></i> Technologie Utilisée</div>
            <div class="grid">
                <div style="background: var(--bg-dark); padding: 25px; border-radius: 12px; border: 1px solid var(--border);">
                    <h3 style="color: var(--primary); margin-bottom: 15px; justify-content: flex-start;">
                        <i class="fas fa-python"></i> Backend
                    </h3>
                    <ul style="color: var(--text-secondary); line-height: 2; list-style: none; padding-left: 0;">
                        <li><i class="fas fa-check" style="color: var(--success);"></i> Python 3.x</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> Flask Web Framework</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> SQLite Database</li>
                    </ul>
                </div>
                <div style="background: var(--bg-dark); padding: 25px; border-radius: 12px; border: 1px solid var(--border);">
                    <h3 style="color: var(--primary); margin-bottom: 15px; justify-content: flex-start;">
                        <i class="fas fa-brain"></i> Machine Learning
                    </h3>
                    <ul style="color: var(--text-secondary); line-height: 2; list-style: none; padding-left: 0;">
                        <li><i class="fas fa-check" style="color: var(--success);"></i> Scikit-learn</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> Gradient Boosting</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> SMOTE Balancing</li>
                    </ul>
                </div>
                <div style="background: var(--bg-dark); padding: 25px; border-radius: 12px; border: 1px solid var(--border);">
                    <h3 style="color: var(--primary); margin-bottom: 15px; justify-content: flex-start;">
                        <i class="fas fa-paint-brush"></i> Frontend
                    </h3>
                    <ul style="color: var(--text-secondary); line-height: 2; list-style: none; padding-left: 0;">
                        <li><i class="fas fa-check" style="color: var(--success);"></i> HTML5 / CSS3</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> JavaScript ES6+</li>
                        <li><i class="fas fa-check" style="color: var(--success);"></i> Font Awesome Icons</li>
                    </ul>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-award"></i> Avantages Clés</div>
            <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
                <div class="alert alert-success">
                    <i class="fas fa-check-circle"></i>
                    <div>
                        <strong>Précision Exceptionnelle</strong><br>
                        Modèle entraîné sur des données réelles avec {model_accuracy*100:.1f}% de précision
                    </div>
                </div>
                <div class="alert alert-info">
                    <i class="fas fa-bolt"></i>
                    <div>
                        <strong>Rapidité d'Exécution</strong><br>
                        Prédictions instantanées en moins de 2 secondes
                    </div>
                </div>
                <div class="alert alert-warning">
                    <i class="fas fa-shield-alt"></i>
                    <div>
                        <strong>Sécurité Maximale</strong><br>
                        Cryptage des données et protection de la vie privée
                    </div>
                </div>
                <div class="alert alert-success">
                    <i class="fas fa-users"></i>
                    <div>
                        <strong>Interface Intuitive</strong><br>
                        Design moderne et facile à utiliser pour tous
                    </div>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-phone"></i> Contact & Support</div>
            <p style="color: var(--text-secondary); text-align: center; font-size: 1.1em; padding: 30px; background: var(--bg-dark); border-radius: 12px;">
                Pour toute question, suggestion ou demande de support, n'hésitez pas à contacter notre équipe.<br>
                <i class="fas fa-envelope" style="color: var(--primary); margin-top: 15px; display: inline-block;"></i>
                <strong style="color: var(--primary);"> support@acadpredict.ai</strong>
            </p>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(about_html)

@app.route('/statistics')
def statistics():
    if 'user_id' not in session:
        return redirect('/login')

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()

    c.execute("SELECT COUNT(*) FROM history WHERE user_id=?", (session['user_id'],))
    total_predictions = c.fetchone()[0]

    c.execute("SELECT prediction, COUNT(*) FROM history WHERE user_id=? GROUP BY prediction", (session['user_id'],))
    pred_counts = dict(c.fetchall())

    c.execute("SELECT AVG(confidence) FROM history WHERE user_id=?", (session['user_id'],))
    avg_confidence = c.fetchone()[0] or 0

    conn.close()

    diplome_count = pred_counts.get('Diplômé', 0)
    encours_count = pred_counts.get('En cours', 0)
    abandon_count = pred_counts.get('Abandon', 0)

    diplome_pct = (diplome_count / total_predictions * 100) if total_predictions > 0 else 0
    encours_pct = (encours_count / total_predictions * 100) if total_predictions > 0 else 0
    abandon_pct = (abandon_count / total_predictions * 100) if total_predictions > 0 else 0

    stats_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-chart-pie"></i> Statistiques - """ + "{{session['username']}}" + """</h2>

            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Vue d'ensemble de vos prédictions</strong><br>
                    Analysez vos résultats et identifiez les tendances dans vos prédictions
                </div>
            </div>

            <div class="stats-grid">
                <div class="stat-card">
                    <i class="fas fa-calculator"></i>
                    <h3>{total_predictions}</h3>
                    <p>Prédictions Totales</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-percentage"></i>
                    <h3>{avg_confidence*100:.1f}%</h3>
                    <p>Confiance Moyenne</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-graduation-cap"></i>
                    <h3>{diplome_count}</h3>
                    <p>Diplômés Prédits</p>
                </div>
                <div class="stat-card">
                    <i class="fas fa-clock"></i>
                    <h3>{encours_count}</h3>
                    <p>En Cours</p>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-chart-bar"></i> Distribution des Résultats</div>

            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px; margin: 20px 0;">
                <div style="margin-bottom: 30px;">
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                        <span style="color: var(--success); font-weight: 600; display: flex; align-items: center; gap: 10px;">
                            <i class="fas fa-graduation-cap"></i> Diplômé
                        </span>
                        <span style="color: var(--success); font-weight: 700; font-size: 1.2em;">
                            {diplome_pct:.1f}% ({diplome_count})
                        </span>
                    </div>
                    <div class="progress-container">
                        <div class="progress-bar" style="width: {diplome_pct}%; background: linear-gradient(90deg, var(--success), var(--success-light));"></div>
                    </div>
                </div>

                <div style="margin-bottom: 30px;">
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                        <span style="color: var(--warning); font-weight: 600; display: flex; align-items: center; gap: 10px;">
                            <i class="fas fa-clock"></i> En Cours
                        </span>
                        <span style="color: var(--warning); font-weight: 700; font-size: 1.2em;">
                            {encours_pct:.1f}% ({encours_count})
                        </span>
                    </div>
                    <div class="progress-container">
                        <div class="progress-bar" style="width: {encours_pct}%; background: linear-gradient(90deg, var(--warning), var(--warning-light));"></div>
                    </div>
                </div>

                <div>
                    <div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 10px;">
                        <span style="color: var(--danger); font-weight: 600; display: flex; align-items: center; gap: 10px;">
                            <i class="fas fa-times-circle"></i> Abandon
                        </span>
                        <span style="color: var(--danger); font-weight: 700; font-size: 1.2em;">
                            {abandon_pct:.1f}% ({abandon_count})
                        </span>
                    </div>
                    <div class="progress-container">
                        <div class="progress-bar" style="width: {abandon_pct}%; background: linear-gradient(90deg, var(--danger), var(--danger-light));"></div>
                    </div>
                </div>
            </div>

            <div class="section-title"><i class="fas fa-trophy"></i> Résumé des Performances</div>
            <div class="features-grid">
                <div class="feature-card">
                    <i class="fas fa-chart-line"></i>
                    <h3>Taux de Réussite</h3>
                    <p style="font-size: 2em; color: var(--success); font-weight: 700; margin: 15px 0;">
                        {diplome_pct:.1f}%
                    </p>
                    <p>Des prédictions indiquent une réussite académique probable</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-exclamation-triangle"></i>
                    <h3>Taux de Risque</h3>
                    <p style="font-size: 2em; color: var(--danger); font-weight: 700; margin: 15px 0;">
                        {abandon_pct:.1f}%
                    </p>
                    <p>Des cas nécessitent une attention particulière et un suivi</p>
                </div>
                <div class="feature-card">
                    <i class="fas fa-balance-scale"></i>
                    <h3>Cas Intermédiaires</h3>
                    <p style="font-size: 2em; color: var(--warning); font-weight: 700; margin: 15px 0;">
                        {encours_pct:.1f}%
                    </p>
                    <p>En progression, nécessitent un accompagnement régulier</p>
                </div>
            </div>

            <div style="text-align: center; margin-top: 40px;">
                <a href="/predict" class="btn" style="max-width: 400px; margin: 0 auto;">
                    <i class="fas fa-plus-circle"></i> Nouvelle Prédiction
                </a>
            </div>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(stats_html)

@app.route('/register', methods=['GET', 'POST'])
def register():
    if request.method == 'POST':
        user, email, pwd = request.form['username'], request.form['email'], hash_pass(request.form['password'])
        try:
            conn = sqlite3.connect(DB_PATH)
            c = conn.cursor()
            c.execute("INSERT INTO users (username, email, password) VALUES (?, ?, ?)", (user, email, pwd))
            conn.commit()
            conn.close()
            return redirect('/login')
        except:
            error_msg = "Erreur : Cet email est déjà utilisé."
            error_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
            <div class='container'>
                <div class='card' style='max-width:500px; margin:auto;'>
                    <div class="alert alert-warning">
                        <i class="fas fa-exclamation-triangle"></i>
                        <div>{error_msg}</div>
                    </div>
                    <h2><i class="fas fa-user-plus"></i> Créer un Compte</h2>
                    <form method='POST'>
                        <div class='form-group'>
                            <label><i class="fas fa-user"></i> Nom d'utilisateur</label>
                            <input type='text' name='username' required placeholder="Votre nom complet">
                        </div>
                        <div class='form-group'>
                            <label><i class="fas fa-envelope"></i> Email</label>
                            <input type='email' name='email' required placeholder="votre.email@exemple.com">
                        </div>
                        <div class='form-group'>
                            <label><i class="fas fa-lock"></i> Mot de passe</label>
                            <input type='password' name='password' required placeholder="Minimum 8 caractères">
                        </div>
                        <button class='btn'><i class="fas fa-check"></i> Créer mon compte</button>
                    </form>
                    <p style="text-align: center; margin-top: 20px; color: var(--text-secondary);">
                        Déjà un compte ? <a href="/login" style="color: var(--primary); text-decoration: none; font-weight: 600;">Se connecter</a>
                    </p>
                </div>
            </div>
            """ + CREDITS_HTML
            return render_template_string(error_html)

    register_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + """
    <div class='container'>
        <div class='card' style='max-width:500px; margin:auto;'>
            <h2><i class="fas fa-user-plus"></i> Créer un Compte</h2>
            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Rejoignez AcadPredict AI</strong><br>
                    Créez votre compte gratuitement et commencez à prédire la réussite académique
                </div>
            </div>
            <form method='POST'>
                <div class='form-group'>
                    <label><i class="fas fa-user"></i> Nom d'utilisateur</label>
                    <input type='text' name='username' required placeholder="Votre nom complet">
                </div>
                <div class='form-group'>
                    <label><i class="fas fa-envelope"></i> Email</label>
                    <input type='email' name='email' required placeholder="votre.email@exemple.com">
                </div>
                <div class='form-group'>
                    <label><i class="fas fa-lock"></i> Mot de passe</label>
                    <input type='password' name='password' required placeholder="Minimum 8 caractères">
                </div>
                <button class='btn'><i class="fas fa-check"></i> Créer mon compte</button>
            </form>
            <p style="text-align: center; margin-top: 20px; color: var(--text-secondary);">
                Déjà un compte ? <a href="/login" style="color: var(--primary); text-decoration: none; font-weight: 600;">Se connecter</a>
            </p>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(register_html)

@app.route('/login', methods=['GET', 'POST'])
def login():
    if request.method == 'POST':
        email, pwd = request.form['email'], hash_pass(request.form['password'])
        conn = sqlite3.connect(DB_PATH)
        c = conn.cursor()
        c.execute("SELECT id, username FROM users WHERE email=? AND password=?", (email, pwd))
        user = c.fetchone()
        conn.close()
        if user:
            session['user_id'] = user[0]
            session['username'] = user[1]
            return redirect('/predict')

        error_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + """
        <div class='container'>
            <div class='card' style='max-width:500px; margin:auto;'>
                <div class="alert alert-warning">
                    <i class="fas fa-exclamation-triangle"></i>
                    <div><strong>Erreur :</strong> Identifiants incorrects</div>
                </div>
                <h2><i class="fas fa-sign-in-alt"></i> Connexion</h2>
                <form method='POST'>
                    <div class='form-group'>
                        <label><i class="fas fa-envelope"></i> Email</label>
                        <input type='email' name='email' required placeholder="votre.email@exemple.com">
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-lock"></i> Mot de passe</label>
                        <input type='password' name='password' required placeholder="Votre mot de passe">
                    </div>
                    <button class='btn'><i class="fas fa-sign-in-alt"></i> Se connecter</button>
                </form>
                <p style="text-align: center; margin-top: 20px; color: var(--text-secondary);">
                    Pas encore de compte ? <a href="/register" style="color: var(--primary); text-decoration: none; font-weight: 600;">S'inscrire</a>
                </p>
            </div>
        </div>
        """ + CREDITS_HTML
        return render_template_string(error_html)

    login_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + """
    <div class='container'>
        <div class='card' style='max-width:500px; margin:auto;'>
            <h2><i class="fas fa-sign-in-alt"></i> Connexion</h2>
            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Bon retour !</strong><br>
                    Connectez-vous pour accéder à vos prédictions
                </div>
            </div>
            <form method='POST'>
                <div class='form-group'>
                    <label><i class="fas fa-envelope"></i> Email</label>
                    <input type='email' name='email' required placeholder="votre.email@exemple.com">
                </div>
                <div class='form-group'>
                    <label><i class="fas fa-lock"></i> Mot de passe</label>
                    <input type='password' name='password' required placeholder="Votre mot de passe">
                </div>
                <button class='btn'><i class="fas fa-sign-in-alt"></i> Se connecter</button>
            </form>
            <p style="text-align: center; margin-top: 20px; color: var(--text-secondary);">
                Pas encore de compte ? <a href="/register" style="color: var(--primary); text-decoration: none; font-weight: 600;">S'inscrire gratuitement</a>
            </p>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(login_html)

@app.route('/logout')
def logout():
    session.clear()
    return redirect('/')

@app.route('/predict', methods=['GET', 'POST'])
def predict():
    if 'user_id' not in session:
        return redirect('/login')

    if request.method == 'POST':
        data = request.get_json()
        def c(v): return float(str(v).replace(',', '.'))

        inputs = np.array([[
            c(data['s2_app']), c(data['s2_grade']), c(data['s1_app']), c(data['s1_grade']),
            c(data['tuition']), c(data['age']), c(data['s2_enr']), c(data['s1_enr']),
            c(data['scholarship']), c(data['s2_eval']), c(data['s1_eval']), c(data['debtor']),
            c(data['gender']), c(data['gdp']), c(data['unemployment'])
        ]])

        scaled = global_scaler.transform(inputs)
        probs = global_model.predict_proba(scaled)[0]
        idx = np.argmax(probs)
        labels = ['Abandon', 'En cours', 'Diplômé']
        result = labels[idx]
        conf = float(np.max(probs))

        conn = sqlite3.connect(DB_PATH)
        c = conn.cursor()
        c.execute("INSERT INTO history (user_id, prediction, confidence) VALUES (?, ?, ?)",
                 (session['user_id'], result, conf))
        conn.commit()
        conn.close()

        return jsonify({'prediction': result, 'confidence': conf})

    predict_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + """
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-chart-line"></i> Nouvelle Prédiction - {{session['username']}}</h2>

            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Remplissez le formulaire</strong><br>
                    Saisissez les informations de l'étudiant pour obtenir une prédiction précise
                </div>
            </div>

            <form id='pForm'>
                <div class='section-title'><i class="fas fa-book-open"></i> Résultats Académiques - Semestre 1</div>
                <div class='grid'>
                    <div class='form-group'>
                        <label><i class="fas fa-check-circle"></i> Unités Validées S1</label>
                        <input type='number' name='s1_app' value='5' min='0' max='20'>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-star"></i> Moyenne S1</label>
                        <input type='text' name='s1_grade' value='12.5' placeholder="0.0 - 20.0">
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-tasks"></i> Évaluations S1</label>
                        <input type='number' name='s1_eval' value='6' min='0'>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-pencil-alt"></i> Unités Inscrites S1</label>
                        <input type='number' name='s1_enr' value='6' min='0'>
                    </div>
                </div>

                <div class='section-title'><i class="fas fa-book-reader"></i> Résultats Académiques - Semestre 2</div>
                <div class='grid'>
                    <div class='form-group'>
                        <label><i class="fas fa-check-circle"></i> Unités Validées S2</label>
                        <input type='number' name='s2_app' value='5' min='0' max='20'>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-star"></i> Moyenne S2</label>
                        <input type='text' name='s2_grade' value='13.0' placeholder="0.0 - 20.0">
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-tasks"></i> Évaluations S2</label>
                        <input type='number' name='s2_eval' value='6' min='0'>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-pencil-alt"></i> Unités Inscrites S2</label>
                        <input type='number' name='s2_enr' value='6' min='0'>
                    </div>
                </div>

                <div class='section-title'><i class="fas fa-user-circle"></i> Profil Social & Économique</div>
                <div class='grid'>
                    <div class='form-group'>
                        <label><i class="fas fa-calendar"></i> Âge à l'inscription</label>
                        <input type='number' name='age' value='20' min='16' max='60'>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-venus-mars"></i> Genre</label>
                        <select name='gender'>
                            <option value='0'>Femme</option>
                            <option value='1'>Homme</option>
                        </select>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-money-check"></i> Frais de Scolarité à Jour</label>
                        <select name='tuition'>
                            <option value='1'>Oui</option>
                            <option value='0'>Non</option>
                        </select>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-award"></i> Boursier</label>
                        <select name='scholarship'>
                            <option value='0'>Non</option>
                            <option value='1'>Oui</option>
                        </select>
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-exclamation-triangle"></i> Situation d'Endettement</label>
                        <select name='debtor'>
                            <option value='0'>Non</option>
                            <option value='1'>Oui</option>
                        </select>
                    </div>
                </div>

                <div class='section-title'><i class="fas fa-globe"></i> Indicateurs Macroéconomiques</div>
                <div class='grid'>
                    <div class='form-group'>
                        <label><i class="fas fa-chart-pie"></i> PIB (GDP) %</label>
                        <input type='text' name='gdp' value='1.2' placeholder="Ex: 1.2">
                    </div>
                    <div class='form-group'>
                        <label><i class="fas fa-briefcase"></i> Taux de Chômage %</label>
                        <input type='text' name='unemployment' value='11.5' placeholder="Ex: 11.5">
                    </div>
                </div>

                <button type='button' onclick='doPredict()' class='btn' style="margin-top: 30px;">
                    <i class="fas fa-brain"></i> Lancer la Prédiction IA
                </button>
            </form>

            <div id='resBox' class='result-box'></div>
        </div>
    </div>
    """ + CREDITS_HTML + """
    <script>
    async function doPredict() {
        const btn = document.querySelector('button[type="button"]');
        const originalHTML = btn.innerHTML;
        btn.innerHTML = '<div class="loading"></div> Analyse en cours...';
        btn.disabled = true;

        try {
            const formData = new FormData(document.getElementById('pForm'));
            const data = Object.fromEntries(formData.entries());

            const response = await fetch('/predict', {
                method: 'POST',
                headers: {'Content-Type': 'application/json'},
                body: JSON.stringify(data)
            });

            const res = await response.json();
            const box = document.getElementById('resBox');
            box.style.display = 'block';

            let icon = '<i class="fas fa-graduation-cap"></i>';
            let message = '';

            if (res.prediction == 'Abandon') {
                icon = '<i class="fas fa-times-circle"></i>';
                message = 'Risque élevé d\\'abandon - Intervention recommandée';
                box.style.borderColor = 'var(--danger)';
                box.style.background = 'rgba(239, 68, 68, 0.1)';
            } else if (res.prediction == 'En cours') {
                icon = '<i class="fas fa-clock"></i>';
                message = 'Étudiant en progression - Suivi nécessaire';
                box.style.borderColor = 'var(--warning)';
                box.style.background = 'rgba(245, 158, 11, 0.1)';
            } else {
                icon = '<i class="fas fa-trophy"></i>';
                message = 'Forte probabilité de réussite !';
                box.style.borderColor = 'var(--success)';
                box.style.background = 'rgba(16, 185, 129, 0.1)';
            }

            box.innerHTML = `
                <h3>${icon} Résultat : ${res.prediction}</h3>
                <p style="font-size: 1.1em; margin: 15px 0;">
                    <i class='fas fa-percentage'></i> Indice de Confiance :
                    <strong style="font-size: 1.3em;">${(res.confidence*100).toFixed(1)}%</strong>
                </p>
                <p style="color: var(--text-secondary); font-style: italic;">
                    ${message}
                </p>
                <div style="margin-top: 20px;">
                    <a href="/dashboard" class="btn btn-secondary" style="max-width: 300px; margin: 0 auto;">
                        <i class="fas fa-history"></i> Voir l'historique
                    </a>
                </div>
            `;

            box.scrollIntoView({ behavior: 'smooth', block: 'nearest' });
        } catch (error) {
            alert('Erreur lors de la prédiction: ' + error);
        } finally {
            btn.innerHTML = originalHTML;
            btn.disabled = false;
        }
    }
    </script>
    """

    return render_template_string(predict_html)

@app.route('/dashboard')
def dashboard():
    if 'user_id' not in session:
        return redirect('/login')

    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.execute("SELECT prediction, confidence, date FROM history WHERE user_id=? ORDER BY date DESC LIMIT 50",
             (session['user_id'],))
    data = c.fetchall()
    conn.close()

    table_rows = "".join([
        f"""<tr>
            <td><i class='fas fa-calendar-alt'></i> {d[2]}</td>
            <td>
                <span class="badge {'badge-success' if d[0]=='Diplômé' else 'badge-warning' if d[0]=='En cours' else 'badge-danger'}">
                    <i class='fas fa-{'graduation-cap' if d[0]=='Diplômé' else 'clock' if d[0]=='En cours' else 'times-circle'}'></i>
                    {d[0]}
                </span>
            </td>
            <td>
                <div class="progress-container" style="display: inline-block; width: 100px; vertical-align: middle; margin-right: 10px;">
                    <div class="progress-bar" style="width: {d[1]*100:.0f}%;"></div>
                </div>
                <strong>{d[1]*100:.1f}%</strong>
            </td>
        </tr>"""
        for d in data
    ])

    dashboard_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-history"></i> Historique des Prédictions - """ + "{{session['username']}}" + """</h2>

            <div class="alert alert-info">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Vos 50 dernières prédictions</strong><br>
                    Consultez l'historique complet de vos analyses
                </div>
            </div>

            <div style="margin: 30px 0; display: flex; gap: 15px; justify-content: space-between; flex-wrap: wrap;">
                <a href="/predict" class="btn btn-success" style="flex: 1; min-width: 250px;">
                    <i class="fas fa-plus-circle"></i> Nouvelle Prédiction
                </a>
                <a href="/statistics" class="btn btn-secondary" style="flex: 1; min-width: 250px;">
                    <i class="fas fa-chart-pie"></i> Voir les Statistiques
                </a>
            </div>

            <table>
                <thead>
                    <tr>
                        <th><i class="fas fa-calendar"></i> Date & Heure</th>
                        <th><i class="fas fa-chart-bar"></i> Résultat</th>
                        <th><i class="fas fa-percentage"></i> Confiance</th>
                    </tr>
                </thead>
                <tbody>
                    {table_rows if table_rows else "<tr><td colspan='3' style='text-align:center; padding: 40px;'><i class='fas fa-inbox' style='font-size: 3em; color: var(--text-muted); display: block; margin-bottom: 15px;'></i><p style='color: var(--text-secondary);'>Aucune prédiction pour le moment</p><a href='/predict' class='btn' style='max-width: 300px; margin: 20px auto;'><i class='fas fa-plus'></i> Créer votre première prédiction</a></td></tr>"}
                </tbody>
            </table>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(dashboard_html)

@app.route('/guide')
def guide():
    guide_html = BASE_CSS + PARTICLES_HTML + NAV_HTML + f"""
    <div class='container'>
        <div class='card'>
            <h2><i class="fas fa-book"></i> Guide d'Utilisation Complet</h2>

            <div class='section-title'><i class="fas fa-info-circle"></i> Introduction à AcadPredict AI</div>
            <p style='color: var(--text-secondary); line-height: 1.9; font-size: 1.1em; padding: 25px; background: var(--bg-dark); border-radius: 12px; border-left: 4px solid var(--primary);'>
                AcadPredict AI est une application innovante qui utilise l'Intelligence Artificielle et le Machine Learning
                pour prédire avec précision la réussite académique des étudiants. Grâce à l'algorithme Gradient Boosting
                et l'analyse de 15 variables clés, notre système peut identifier les étudiants à risque et ceux qui ont
                de fortes chances de réussir leur parcours académique.
            </p>

            <div class='section-title'><i class="fas fa-rocket"></i> Guide Pas à Pas</div>
            <div class="features-grid">
                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">1</div>
                    <h3>Créer un Compte</h3>
                    <p>Inscrivez-vous gratuitement en fournissant votre nom, email et un mot de passe sécurisé. Vos données sont cryptées avec SHA-256.</p>
                </div>

                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">2</div>
                    <h3>Se Connecter</h3>
                    <p>Accédez à votre espace personnel avec vos identifiants. Votre session est sécurisée et vos données restent privées.</p>
                </div>

                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">3</div>
                    <h3>Remplir le Formulaire</h3>
                    <p>Saisissez les 15 variables requises : résultats académiques (S1 & S2), profil social et indicateurs économiques.</p>
                </div>

                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">4</div>
                    <h3>Obtenir la Prédiction</h3>
                    <p>Cliquez sur "Lancer la Prédiction IA" et recevez instantanément le résultat avec un indice de confiance précis.</p>
                </div>

                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">5</div>
                    <h3>Consulter l'Historique</h3>
                    <p>Toutes vos prédictions sont automatiquement sauvegardées et accessibles dans votre tableau de bord personnel.</p>
                </div>

                <div class="feature-card">
                    <div style="background: linear-gradient(135deg, var(--primary), var(--secondary)); color: white; width: 50px; height: 50px; border-radius: 50%; display: flex; align-items: center; justify-content: center; font-size: 1.5em; font-weight: 700; margin-bottom: 15px;">6</div>
                    <h3>Analyser les Statistiques</h3>
                    <p>Visualisez la distribution de vos résultats et identifiez les tendances grâce à des graphiques interactifs.</p>
                </div>
            </div>

            <div class='section-title'><i class="fas fa-database"></i> Les 15 Variables Analysées</div>

            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px; margin: 20px 0;">
                <h3 style="color: var(--primary); margin-bottom: 20px; display: flex; align-items: center; gap: 10px;">
                    <i class="fas fa-book-open"></i> Performances Académiques (8 variables)
                </h3>
                <div class="grid" style="grid-template-columns: 1fr 1fr;">
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-check-circle" style="color: var(--success);"></i>
                        <strong style="color: var(--text-primary);"> Unités Validées S1/S2</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Nombre de cours réussis</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-star" style="color: var(--warning);"></i>
                        <strong style="color: var(--text-primary);"> Moyennes S1/S2</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Notes moyennes obtenues</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-tasks" style="color: var(--info);"></i>
                        <strong style="color: var(--text-primary);"> Évaluations S1/S2</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Nombre d'examens passés</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-pencil-alt" style="color: var(--secondary);"></i>
                        <strong style="color: var(--text-primary);"> Unités Inscrites S1/S2</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Cours auxquels l'étudiant est inscrit</p>
                    </div>
                </div>
            </div>

            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px; margin: 20px 0;">
                <h3 style="color: var(--primary); margin-bottom: 20px; display: flex; align-items: center; gap: 10px;">
                    <i class="fas fa-user-circle"></i> Profil Socio-Économique (5 variables)
                </h3>
                <div class="grid" style="grid-template-columns: 1fr 1fr;">
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-calendar" style="color: var(--primary);"></i>
                        <strong style="color: var(--text-primary);"> Âge à l'inscription</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Âge de l'étudiant lors de son inscription</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-venus-mars" style="color: var(--secondary);"></i>
                        <strong style="color: var(--text-primary);"> Genre</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Homme ou Femme</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-money-check" style="color: var(--success);"></i>
                        <strong style="color: var(--text-primary);"> Frais de Scolarité</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Statut de paiement des frais</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-award" style="color: var(--warning);"></i>
                        <strong style="color: var(--text-primary);"> Bourse</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Bénéficiaire d'une bourse ou non</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px; margin-bottom: 10px;">
                        <i class="fas fa-exclamation-triangle" style="color: var(--danger);"></i>
                        <strong style="color: var(--text-primary);"> Endettement</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Situation de dette financière</p>
                    </div>
                </div>
            </div>

            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px; margin: 20px 0;">
                <h3 style="color: var(--primary); margin-bottom: 20px; display: flex; align-items: center; gap: 10px;">
                    <i class="fas fa-globe"></i> Contexte Macroéconomique (2 variables)
                </h3>
                <div class="grid" style="grid-template-columns: 1fr 1fr;">
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px;">
                        <i class="fas fa-chart-pie" style="color: var(--info);"></i>
                        <strong style="color: var(--text-primary);"> PIB (GDP)</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Produit Intérieur Brut du pays</p>
                    </div>
                    <div style="padding: 15px; background: var(--bg-card); border-radius: 8px;">
                        <i class="fas fa-briefcase" style="color: var(--warning);"></i>
                        <strong style="color: var(--text-primary);"> Taux de Chômage</strong>
                        <p style="color: var(--text-secondary); margin: 5px 0 0 0; font-size: 0.95em;">Pourcentage de chômage national</p>
                    </div>
                </div>
            </div>

            <div class='section-title'><i class="fas fa-chart-line"></i> Interprétation des Résultats</div>
            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px;">
                <p style="color: var(--text-secondary); margin-bottom: 25px; font-size: 1.05em;">
                    L'IA peut prédire 3 résultats possibles avec un indice de confiance :
                </p>
                <div style="display: flex; flex-direction: column; gap: 20px;">
                    <div style="border-left: 4px solid var(--success); padding: 20px; background: rgba(16, 185, 129, 0.1); border-radius: 8px;">
                        <h3 style="color: var(--success); display: flex; align-items: center; gap: 10px; margin-bottom: 10px;">
                            <i class="fas fa-trophy"></i> Diplômé
                        </h3>
                        <p style="color: var(--text-secondary); line-height: 1.7;">
                            L'étudiant a de <strong style="color: var(--success);">fortes chances de terminer avec succès</strong> son cursus.
                            Les facteurs académiques et socio-économiques sont favorables. Continuez le bon travail !
                        </p>
                    </div>
                    <div style="border-left: 4px solid var(--warning); padding: 20px; background: rgba(245, 158, 11, 0.1); border-radius: 8px;">
                        <h3 style="color: var(--warning); display: flex; align-items: center; gap: 10px; margin-bottom: 10px;">
                            <i class="fas fa-hourglass-half"></i> En cours
                        </h3>
                        <p style="color: var(--text-secondary); line-height: 1.7;">
                            L'étudiant est en progression mais <strong style="color: var(--warning);">nécessite un suivi régulier</strong>.
                            Certains indicateurs montrent des signes de fragilité. Un accompagnement personnalisé est recommandé.
                        </p>
                    </div>
                    <div style="border-left: 4px solid var(--danger); padding: 20px; background: rgba(239, 68, 68, 0.1); border-radius: 8px;">
                        <h3 style="color: var(--danger); display: flex; align-items: center; gap: 10px; margin-bottom: 10px;">
                            <i class="fas fa-exclamation-triangle"></i> Abandon
                        </h3>
                        <p style="color: var(--text-secondary); line-height: 1.7;">
                            Risque élevé d'abandon du cursus. <strong style="color: var(--danger);">Intervention urgente recommandée</strong>.
                            Les facteurs de risque sont nombreux. Un plan d'action doit être mis en place immédiatement.
                        </p>
                    </div>
                </div>
            </div>

            <div class='section-title'><i class="fas fa-lightbulb"></i> Conseils d'Utilisation</div>
            <div class="alert alert-info" style="margin: 20px 0;">
                <i class="fas fa-info-circle"></i>
                <div>
                    <strong>Astuce :</strong> Pour obtenir les prédictions les plus précises, assurez-vous de saisir des données exactes et à jour.
                </div>
            </div>
            <div class="alert alert-success" style="margin: 20px 0;">
                <i class="fas fa-check-circle"></i>
                <div>
                    <strong>Recommandation :</strong> Effectuez des prédictions régulières pour suivre l'évolution des étudiants dans le temps.
                </div>
            </div>
            <div class="alert alert-warning" style="margin: 20px 0;">
                <i class="fas fa-exclamation-triangle"></i>
                <div>
                    <strong>Important :</strong> Les prédictions sont des outils d'aide à la décision, pas des certitudes absolues.
                </div>
            </div>

            <div class='section-title'><i class="fas fa-question-circle"></i> Foire Aux Questions</div>
            <div style="background: var(--bg-dark); padding: 30px; border-radius: 12px;">
                <div style="margin-bottom: 25px;">
                    <h3 style="color: var(--primary); margin-bottom: 10px;">
                        <i class="fas fa-question"></i> Quelle est la précision du modèle ?
                    </h3>
                    <p style="color: var(--text-secondary); padding-left: 30px;">
                        Notre modèle Gradient Boosting atteint une précision de <strong>{model_accuracy*100:.1f}%</strong> sur les données de test,
                        ce qui en fait un outil fiable pour la prédiction académique.
                    </p>
                </div>

                <div style="margin-bottom: 25px;">
                    <h3 style="color: var(--primary); margin-bottom: 10px;">
                        <i class="fas fa-question"></i> Mes données sont-elles sécurisées ?
                    </h3>
                    <p style="color: var(--text-secondary); padding-left: 30px;">
                        Oui, absolument. Vos mots de passe sont cryptés avec SHA-256 et vos données sont stockées localement
                        dans une base SQLite sécurisée. Nous ne partageons jamais vos informations.
                    </p>
                </div>

                <div style="margin-bottom: 25px;">
                    <h3 style="color: var(--primary); margin-bottom: 10px;">
                        <i class="fas fa-question"></i> Puis-je exporter mes prédictions ?
                    </h3>
                    <p style="color: var(--text-secondary); padding-left: 30px;">
                        Oui, vous pouvez consulter et analyser toutes vos prédictions dans la section Historique et Statistiques.
                    </p>
                </div>

                <div>
                    <h3 style="color: var(--primary); margin-bottom: 10px;">
                        <i class="fas fa-question"></i> Combien de prédictions puis-je faire ?
                    </h3>
                    <p style="color: var(--text-secondary); padding-left: 30px;">
                        Il n'y a aucune limite ! Vous pouvez effectuer autant de prédictions que vous le souhaitez.
                        Toutes sont automatiquement sauvegardées dans votre historique personnel.
                    </p>
                </div>
            </div>

            <div style="text-align: center; margin-top: 50px;">
                <h3 style="color: var(--primary); margin-bottom: 20px;">
                    <i class="fas fa-rocket"></i> Prêt à commencer ?
                </h3>
    """ + """
                {% if session.get('user_id') %}
                    <a href="/predict" class="btn" style="max-width: 400px; margin: 0 auto;">
                        <i class="fas fa-chart-line"></i> Lancer une Prédiction
                    </a>
                {% else %}
                    <a href="/register" class="btn" style="max-width: 400px; margin: 0 auto;">
                        <i class="fas fa-user-plus"></i> Créer un Compte Gratuit
                    </a>
                {% endif %}
    """ + """
            </div>
        </div>
    </div>
    """ + CREDITS_HTML

    return render_template_string(guide_html)

# =================================================================
# 5. LANCEMENT DU SITE
# =================================================================
if __name__ == "__main__":
    init_db()
    !fuser -k 5000/tcp
    print("\n" + "="*60)
    print("🚀 AcadPredict AI - Prédiction Académique par Intelligence Artificielle")
    print("="*60)
    print(f"✅ Modèle IA chargé avec {model_accuracy*100:.1f}% de précision")
    print(f"✅ Base de données initialisée")
    print(f"✅ {len(features_list)} variables d'analyse configurées")
    print("\n📍 Interface accessible ici :")
    print(eval_js("google.colab.kernel.proxyPort(5000)"))
    print("\n" + "="*60)
    app.run(host="0.0.0.0", port=5000, debug=False)


⏳ Entraînement du modèle en cours...
✅ Modèle IA opérationnel !

🚀 AcadPredict AI - Prédiction Académique par Intelligence Artificielle
✅ Modèle IA chargé avec 74.7% de précision
✅ Base de données initialisée
✅ 15 variables d'analyse configurées

📍 Interface accessible ici :
https://5000-m-s-9rvaf9srymxd-b.us-west3-0.prod.colab.dev

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:23:56] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:04] "GET /favicon.ico HTTP/1.1" 404 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:31] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:33] "GET /features HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:40] "GET /guide HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:47] "GET /login HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:24:50] "GET /register HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:25:10] "POST /register HTTP/1.1" 302 -
INFO:werkzeug:127.0.0.1 - - [13/Feb/2026 17:25:11] "GET /login HTTP/1.1" 200 -
INF